[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 07](README.md)

# Híbrido MPI + OpenMP

**Tema:** 07 · **Sesiones:** 31, 34 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cómo mapear procesos e hilos a nodos, NUMA y núcleos sin sobresuscripción?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** El diseño híbrido reparte paralelismo entre nodos y dentro de cada nodo. El producto ranks×hilos debe corresponder a recursos físicos y afinidad.

**Prerrequisitos.**

- MPI, OpenMP y un modelo de acelerador.
- Afinidad, escalabilidad y lectura de perfiles.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Relacionar ranks, hilos, núcleos y dominios NUMA.
- Interpretar niveles de `MPI_Init_thread`.
- Diseñar afinidad y first-touch reproducibles.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

MPI distribuye memoria entre procesos; OpenMP explota memoria compartida dentro del proceso.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

El nivel de soporte de hilos limita qué hilos pueden invocar MPI.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

ranks × threads debe corresponder a la asignación y la afinidad debe evitar migraciones no controladas.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- afinidad — vínculo entre trabajo y recursos físicos
- sobresuscripción — más entidades ejecutables que recursos asignados
- perfil — atribución del tiempo a regiones o fases


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Topologia Hibrida

![Nodos con ranks, hilos y GPU local](../../images/topologia-hibrida.svg)

**Cómo leerlo.** Recorre la jerarquía de afuera hacia adentro. El mapeo correcto conserva afinidad local y evita asignar accidentalmente varios ranks al mismo dispositivo.

### Fork Join

![Región serial que crea y reúne trabajadores](../../images/fork-join.svg)

**Cómo leerlo.** La región posterior al join solo puede consumir el resultado cuando todos los trabajadores necesarios terminaron y publicaron sus parciales.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "07"
NOTEBOOK = "07_hibrido/01_mpi_openmp.ipynb"
assert (ROOT / "curso" / "notebooks" / "07_hibrido" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Mapa de recursos

**Situación.** Se genera una asignación simple por nodo y se comprueba que no exceda núcleos.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
nodes, cores_per_node, ranks_per_node, threads_per_rank = 2, 32, 4, 8
assert ranks_per_node * threads_per_rank <= cores_per_node
mapping = []
for node in range(nodes):
    for local_rank in range(ranks_per_node):
        first_core = local_rank * threads_per_rank
        mapping.append((node, local_rank, tuple(range(first_core, first_core+threads_per_rank))))
for row in mapping: print(row)


### Explicación del resultado

En hardware con SMT o NUMA, la política se adapta y se registra mediante herramientas del runtime/planificador.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Soporte de hilos MPI

**Situación.** Se ordenan los niveles y se verifica una solicitud.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
levels = {"SINGLE": 0, "FUNNELED": 1, "SERIALIZED": 2, "MULTIPLE": 3}
requested, provided = "FUNNELED", "SERIALIZED"
assert levels[provided] >= levels[requested]
for name, value in levels.items(): print(value, name)
print("solicitado", requested, "provisto", provided)


### Lectura razonada

El programa aborta o cambia de estrategia si el nivel provisto es inferior al solicitado.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué configuración mantendría constante el total de núcleos al comparar más ranks con más hilos?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Elegir ranks por NUMA y hilos por rank.
2. Registrar `OMP_PLACES`, `OMP_PROC_BIND` y opciones Slurm.
3. Comparar MPI puro, OpenMP puro e híbrido con igual total de núcleos.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Usar `MPI_THREAD_MULTIPLE` sin necesitarlo ni medir costo.
- Olvidar que bibliotecas internas pueden crear hilos.
- Comparar configuraciones con distinto total de recursos.


## Criterios de aceptación

- No hay sobresuscripción involuntaria.
- Nivel MPI provisto comprobado.
- Afinidad y first-touch documentados.


## Síntesis

- La pregunta que debes poder responder es: **¿Cómo mapear procesos e hilos a nodos, NUMA y núcleos sin sobresuscripción?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Entornos de clúster](../../../topicos_avanzados/ENTORNOS_CLUSTER.md)
- [Planeación híbrida](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 07](README.md)
